In [2]:
# 1. 패키지 설치 및 구글 드라이브 마운트
!pip install pyreadstat

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pyreadstat

# 2. 파일 경로 설정
DATA_DIR = '/content/drive/MyDrive/hdac_data'

# 3. 타겟 변수 불러오기
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn', 'HEI', 'incm', 'edu', 'BD1_11', 'pa_aerobic'
]

df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)

# 식생활평가지수 변수명 표준화
if 'HEI' in df.columns:
    df['HE_khei'] = df['HEI']

print(f"✅ 데이터 로드 완료! 초기 전체 표본 수: {len(df):,}명")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 19.3 MB/s  0:00:00
Mounted at /content/drive
✅ 데이터 로드 완료! 초기 전체 표본 수: 6,997명


In [3]:
# 1. 1인 가구 재코딩 (1: 1인가구, 0: 다인가구, NaN: 무응답/기타)
df['genertn_rc'] = df['genertn'].apply(
    lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan)
)

# 2. 단계별 조건 정의
cond_adult = df['age'] >= 20  # 20세 이상 성인
cond_wt = df['wt_ntr'].notnull()  # 가중치 유효
cond_exp = df['genertn_rc'].notnull()  # 노출 유효
cond_out = df['HE_khei'].notnull()  # 결과 유효

# 3. 행을 지우지 않고 도메인 분석용 subpop 변수 생성 (1: 최종 분석 대상, 0: 제외 대상)
df['subpop'] = np.where(
    cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0
)

# 4. B-1 흐름도용 인원 산출 및 출력
n_total = len(df)
n_adult = cond_adult.sum()
n_ex_age = n_total - n_adult

n_step2 = (cond_adult & cond_wt).sum()
n_ex_wt = n_adult - n_step2

n_step3 = (cond_adult & cond_wt & cond_exp).sum()
n_ex_exp = n_step2 - n_step3

n_final = int(df['subpop'].sum())
n_ex_out = n_step3 - n_final

print('=' * 50)
print(' [B-1 분석 대상 선정 단계별 흐름도 데이터] ')
print('=' * 50)
print(f'1단계 초기 전체 표본: {n_total:,}명')
print(f'  - 제외 (미성년자 <20세): {n_ex_age:,}명')
print(f'2단계 20세 이상 성인: {n_adult:,}명')
print(f'  - 제외 (영양조사 가중치 결측): {n_ex_wt:,}명')
print(f'3단계 가중치 적용 가능자: {n_step2:,}명')
print(f'  - 제외 (1인 가구 여부 무응답/결측): {n_ex_exp:,}명')
print(f'4단계 노출 유효 대상자: {n_step3:,}명')
print(f'  - 제외 (KHEI 식생활평가지수 결측): {n_ex_out:,}명')
print(f'✅ 최종 분석 대상 (subpop == 1): {n_final:,}명')
print('=' * 50)

 [B-1 분석 대상 선정 단계별 흐름도 데이터] 
1단계 초기 전체 표본: 6,997명
  - 제외 (미성년자 <20세): 995명
2단계 20세 이상 성인: 6,002명
  - 제외 (영양조사 가중치 결측): 174명
3단계 가중치 적용 가능자: 5,828명
  - 제외 (1인 가구 여부 무응답/결측): 9명
4단계 노출 유효 대상자: 5,819명
  - 제외 (KHEI 식생활평가지수 결측): 3명
✅ 최종 분석 대상 (subpop == 1): 5,816명


In [4]:
# 최종 분석 대상(subpop == 1)만 추출하여 통계량 계산
df_sub = df[df['subpop'] == 1.0].copy()

groups = {
    'Total (전체)': df_sub,
    '1-Person (1인 가구)': df_sub[df_sub['genertn_rc'] == 1.0],
    'Multi-Person (다인 가구)': df_sub[df_sub['genertn_rc'] == 0.0],
}


# 가중 평균 및 SE 계산 함수
def get_weighted_stats(val, wt):
  v = np.array(val)
  w = np.array(wt)
  mean = np.sum(v * w) / np.sum(w)
  var = np.sum(w * (v - mean) ** 2) / np.sum(w)
  se = np.sqrt(var / len(v))
  return mean, se


print('=' * 60)
print(' [ Table 1 가중치 적용 vs 비가중 통계량 동시 산출 ]')
print('=' * 60)

for g_name, g_df in groups.items():
  unw_n = len(g_df)
  print(f'\n📌 [{g_name}] (비가중 대상자 수 = {unw_n:,}명)')

  # 연속형: 연령, KHEI
  for c in ['age', 'HE_khei']:
    wm, wse = get_weighted_stats(g_df[c], g_df['wt_ntr'])
    print(
        f'  - {c}: 가중 평균 = {wm:.2f} ± {wse:.2f} | 비가중 평균 ='
        f' {g_df[c].mean():.2f}'
    )

  # 범주형 비율 확인 (비가중 % vs 가중 %)
  print('\n  --- 범주형 공변량 (비가중 % vs 가중 %) ---')
  for cat in ['sex', 'incm', 'edu', 'BD1_11', 'pa_aerobic']:
    print(f'  * [{cat}] 결측수: {g_df[cat].isnull().sum()}명')
    unw_pct = (g_df[cat].value_counts(normalize=True, dropna=False) * 100).round(
        2
    )
    w_sum = g_df.groupby(cat, dropna=False)['wt_ntr'].sum()
    w_pct = ((w_sum / g_df['wt_ntr'].sum()) * 100).round(2)

    comp_df = pd.DataFrame({'비가중 %': unw_pct, '가중 %': w_pct})
    print(comp_df)
    print('-' * 40)

 [ Table 1 가중치 적용 vs 비가중 통계량 동시 산출 ]

📌 [Total (전체)] (비가중 대상자 수 = 5,816명)
  - age: 가중 평균 = 50.11 ± 0.22 | 비가중 평균 = 54.29
  - HE_khei: 가중 평균 = 58.78 ± 0.17 | 비가중 평균 = 60.26

  --- 범주형 공변량 (비가중 % vs 가중 %) ---
  * [sex] 결측수: 0명
     비가중 %   가중 %
sex              
1.0  42.57  49.76
2.0  57.43  50.24
----------------------------------------
  * [incm] 결측수: 21명
      비가중 %   가중 %
incm              
1.0   24.81  24.28
2.0   24.98  24.86
3.0   25.02  25.28
4.0   24.83  25.22
NaN    0.36   0.35
----------------------------------------
  * [edu] 결측수: 148명
     비가중 %   가중 %
edu              
1.0  16.30  11.35
2.0  10.09   7.85
3.0  31.67  33.09
4.0  39.39  45.66
NaN   2.54   2.06
----------------------------------------
  * [BD1_11] 결측수: 0명
        비가중 %   가중 %
BD1_11              
1.0     18.66  16.71
2.0     19.29  19.29
3.0     11.26  11.69
4.0     20.22  22.35
5.0     13.22  14.97
6.0      4.83   5.42
8.0     10.11   7.64
9.0      2.41   1.95
----------------------------------------
  * [pa_a